# CMPE 401 Project 1 — VisDrone YOLOv11 (Colab)

Runs the **Part I baseline** (YOLOv11s, 640px) on a free Colab GPU, and serves as the base for the VRAM-heavy Part III/V runs (larger models, `imgsz=1280`) that don't fit on the RTX 3060 laptop. It clones the repo, installs deps, downloads + converts VisDrone, then trains/evaluates with the same scripts used locally.

**Steps:**
1. **Set the runtime to a GPU:** Runtime → Change runtime type → GPU (T4). Cell 2 should then print `CUDA available: True`.
2. Run cells 1→3 in order. Cell 3 should finish with ~457k boxes kept (same as local).
3. Run cell 4 to train (~2.5–4 h). **Keep this tab open and active** — free Colab disconnects after ~90 min idle. If dropped, re-run cell 4 with `--resume` appended to continue from the last checkpoint.
4. Run cells 5–7. Cell 7 copies results to Google Drive so they survive the session.

In [ ]:
# 1. Clone your repo (replace with your GitHub URL after you create it)
REPO_URL = "https://github.com/eijikuinbii/CMPE401-YOLOv11"
!git clone $REPO_URL project
%cd project

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Download + convert VisDrone-DET
!python scripts/download_visdrone.py --root datasets/VisDrone
!python scripts/convert_visdrone.py --root datasets/VisDrone

In [ ]:
# 4. Train — Part I baseline (YOLOv11s, 640px). T4 has 16 GB so batch 16 is fine.
# --patience 50 early-stops if val stops improving, so 100 is a safe upper bound.
# If you later run the Part III high-res experiment, change to:
#   --model yolo11m.pt --imgsz 1280 --batch 16 --name exp_m_1280
!python scripts/train.py --model yolo11s.pt --imgsz 640 --batch 16 --epochs 100 --name baseline_s_640

In [ ]:
# 5. Evaluate on val and test-dev
!python scripts/evaluate.py --weights results/runs/baseline_s_640/weights/best.pt --split val --imgsz 640
!python scripts/evaluate.py --weights results/runs/baseline_s_640/weights/best.pt --split test --imgsz 640

In [ ]:
# 6. Plots + summary table
!python scripts/plot_curves.py results/runs/baseline_s_640
!python scripts/summarize_runs.py results/runs/baseline_s_640

In [ ]:
# 7. Save weights + results back to Google Drive so they survive the session.
# Run this AFTER training; re-run it any time to update the Drive copy.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cmpe401_project1
!cp -r results/runs/baseline_s_640 /content/drive/MyDrive/cmpe401_project1/
print('Saved to Drive: MyDrive/cmpe401_project1/baseline_s_640')